In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns

# Continuous Distributions

A continuous distribution is just like a discrete one, except the range of possible outcomes is continous instead of having a limited, discrete set of possible outcomes. For example, the possible outcome events might be any floating point number in the range [0,1] or in the range[-inf, +inf]

Instead of a probability mass function, with the sum of all probabilities adding up to 1, we have a probability density function, with a "density" value for each outcome. And the area under the curve totals 1.

### Mixture of two betas
Let's start with a bit of a mystery distribution. Most likely you haven't seen a mixture of two beta distributions. But we can investigate it empirically.



In [ ]:
def sample_mix_of_two_betas(alpha1=1, beta1=5, alpha2=5, beta2=1, mix=0.5, n=1000):
    which = np.random.choice([1, 2], size=n, p=[mix, 1 - mix])
    b1 = np.random.beta(alpha1, beta1, n)
    b2 = np.random.beta(alpha2, beta2, n)
    df = pd.DataFrame({"outcome": np.where(which == 1, b1, b2)})
    return df

In [ ]:
sample_mix_of_two_betas()

In [ ]:
data = sample_mix_of_two_betas(n=100000)
# divide into 100 bins and compute counts within each bin
bins = np.arange(0, 1.01, 0.01)  # Creates an array from 0 to 1 with a step of 0.01
data["bin"], bin_ranges = pd.cut(data["outcome"], bins=bins, retbins=True)
data

In [ ]:
counts = data.groupby("bin", observed=False).size()
counts

In [ ]:
counts / counts.sum()

In [ ]:
def plot_continuous(
    data, ax1, num_bins=100, label_precision=2, min_val=0, max_val=1, max_prob=None
):
    # Create the bar plot on the first y-axis

    # filter out the data outside the min_val and max_val range
    data = data[(data["outcome"] >= min_val) & (data["outcome"] <= max_val)]

    # define the bin edges
    bin_edges = np.linspace(min_val, max_val, num_bins + 1)
    data["bin"], bin_ranges = pd.cut(data["outcome"], bins=bin_edges, retbins=True)
    counts = data.groupby("bin", observed=False).size()
    probs = counts / counts.sum()
    # use the left edge of each bin_range as the x-axis labels, rounded to the nearest 2 decimal places
    left_edges = bin_ranges[:-1]
    # round the labels to 2 decimal places
    labels = np.round(left_edges, label_precision)

    sns.barplot(x=labels, y=probs, ax=ax1)
    # set maximum probability value if provided
    if max_prob:
        ax1.set_ylim(0, max_prob)

    # Only show every 10th label on the x-axis, for better readability
    for ind, label in enumerate(ax1.get_xticklabels()):
        if ind % 10 == 0:  # every 10th label is kept
            label.set_visible(True)
        else:
            label.set_visible(False)
        # Set y-axis label to "Probability"

    ax1.set_ylabel("Probability")

In [ ]:
plot_continuous(sample_mix_of_two_betas(n=5000000), plt.gca())

In [ ]:
plot_continuous(sample_mix_of_two_betas(2, 5, 7, 3, mix=0.6, n=5000000), plt.gca())

## Uniform distribution

In [ ]:
def sample_uniform(low=0.0, high=1.0, n=1000):
    return pd.DataFrame({"outcome": np.random.uniform(low=low, high=high, size=n)})

In [ ]:
plot_continuous(sample_uniform(n=10000000), plt.gca(), num_bins=50)

In [ ]:
sample_uniform(low=1, high=11, n=20)

In [ ]:
df = sample_uniform(low=1, high=11, n=10000000)
# output minimum and maximum values of the "outcome" column
df["outcome"].min(), df["outcome"].max()

In [ ]:
plot_continuous(
    sample_uniform(low=1, high=11, n=10000000),
    plt.gca(),
    num_bins=50,
    min_val=1,
    max_val=11,
)

## Normal distribution

The normal distribution is also known as the Gaussian distribution.
It is characterized by two parameters:
- The mean μ determines the location of the peak of the distribution and the standard deviation σ determines the width of the distribution.
- The normal distribution is symmetric about the mean μ.

The density function is
$ f(x | \mu, \sigma) = \frac{1}{\sigma \sqrt{2 \pi}} e^{-\frac{(x - \mu)^2}{2 \sigma^2}} $
where:
- x is any real number, positive or negative.
- μ is the mean of the distribution.
- σ is the standard deviation of the distribution.
- π is a mathematical constant, approximately equal to 3.14159.

In [ ]:
def sample_normal(mu=0, sigma=1, n=1000):
    return pd.DataFrame({"outcome": np.random.normal(loc=mu, scale=sigma, size=n)})

In [ ]:
sample_normal(n=20)

In [ ]:
plot_continuous(
    sample_normal(n=10000000), plt.gca(), min_val=-5, max_val=5, label_precision=2
)

In [ ]:
# make 9 small multiples with different mu and sigma values for the normal distribution
fig, axs = plt.subplots(2, 3, figsize=(12, 12))
for i, mu in enumerate([0, 2]):
    for j, sigma in enumerate([1, 3, 5]):
        plot_continuous(
            sample_normal(mu=mu, sigma=sigma, n=1000000),
            axs[i, j],
            min_val=-10,
            max_val=10,
            max_prob=0.2,
            num_bins=50,
            label_precision=1,
        )
        axs[i, j].set_title(f"mu={mu}, sigma={sigma}")

## Exponential distribution
The exponential distribution is a continuous probability distribution that describes the time between events in a Poisson process.
It is defined by a single parameter, lambda (λ), which is the rate parameter. The rate parameter describes the number of events that occur in a unit of time.

The probability density function (pdf) of the exponential distribution is given by:
$$ f(x; \lambda) = 
\begin{cases} 
\lambda e^{-\lambda x} & \text{for } x \geq 0, \\
0 & \text{otherwise}
\end{cases} 
$$

In [ ]:
def sample_exponential(lam=1, n=1000):
    return pd.DataFrame({"outcome": np.random.exponential(scale=1 / lam, size=n)})

In [ ]:
sample_exponential(n=20)

In [ ]:
plot_continuous(
    sample_exponential(lam=1, n=10000000),
    plt.gca(),
    min_val=0,
    max_val=10,
    label_precision=1,
)

In [ ]:
# 3 small multiples with lambda values of 1, 2, and 5
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for i, lam in enumerate([1, 2, 5]):
    plot_continuous(
        sample_exponential(lam=lam, n=1000000),
        axs[i],
        min_val=0,
        max_val=10,
        max_prob=0.8,
        num_bins=50,
        label_precision=1,
    )
    axs[i].set_title(f"lambda={lam}")